# Notebook 02: Data Preprocessing

**Purpose:** Apply all data quality fixes documented in `src/preprocessing.py`. Every fix is a function call — no inline cleaning logic here. This keeps the notebook readable and ensures inference-time code uses the identical functions.

**Output:** `data/interim/cleaned_pre_split.csv` — fully cleaned, not yet split.

**Why save pre-split?** Any future analyst can re-split with a different random seed. Saving post-split would require maintaining the exact same seed to reproduce the experiment.

**Data Quality Issues Resolved (9 total):**

| # | Column | Issue | Resolution |
|---|--------|-------|------------|
| 1 | has_momo_account | 7+ inconsistent encodings | standardize_boolean() |
| 2 | annual_revenue_ghs | Mixed currency prefixes | clean_currency() |
| 3 | owner_gender | Inconsistent M/Male/male/F | standardize_gender() |
| 4 | gra_tin | Empty/PENDING → create has_tin | create_has_tin() |
| 5 | owner_age | Ages 14, 15 — legally invalid | validate_owner_age() |
| 6 | previous_default | Many missing — missing ≠ No | Two features: has_history + clean |
| 7 | credit_bureau_score | Many missing — absence = signal | has_score flag + sector-median impute |
| 8 | bank_account_tenure_months | Suspicious zeros | tenure_is_zero_flag (do not null) |
| 9 | application_date | Raw date string | Cyclical month + year features |

In [1]:
# ── CELL 1: Environment setup ──────────────────────────────────────────────────
import sys, os

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/stanbic_sme_credit'
except ImportError:
    IN_COLAB = False
    _search = os.getcwd()
    BASE_PATH = _search
    for _ in range(3):
        if os.path.isdir(os.path.join(BASE_PATH, 'src')):
            break
        BASE_PATH = os.path.dirname(BASE_PATH)

sys.path.insert(0, os.path.join(BASE_PATH, 'src'))
env_str = 'Google Colab' if IN_COLAB else 'Local Jupyter'
print(f'Environment : {env_str}')
print(f'Base path   : {BASE_PATH}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import our reusable module
from preprocessing import (
    clean_dataframe, engineer_features, drop_non_model_columns,
    print_quality_log, DATA_QUALITY_LOG,
    ALL_MODEL_FEATURES, FAIRNESS_COLS, TARGET
)

pd.set_option('display.max_columns', 50)
print('Environment ready. preprocessing.py loaded.')

Mounted at /content/drive
Environment : Google Colab
Base path   : /content/drive/MyDrive/stanbic_sme_credit
Environment ready. preprocessing.py loaded.


In [2]:
# ── CELL 2: Load raw data ──────────────────────────────────────────────────────
RAW_PATH = f'{BASE_PATH}/data/raw/ds-sme_loan_applications_stanbic_gh.csv'
df_raw = pd.read_csv(RAW_PATH, low_memory=False)

print(f'Raw data loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Target distribution (raw):')
print(df_raw[TARGET].value_counts(dropna=False))

Raw data loaded: 3,037 rows x 29 columns
Target distribution (raw):
loan_default
0    2634
1     403
Name: count, dtype: int64


## Step 1: Document Data Quality Issues

In [3]:
# ── CELL 3: Print full data quality log ───────────────────────────────────────
# This is the formal documentation required by the PDF:
# "Document every data quality issue found and how it was handled"
print_quality_log()

DATA QUALITY ISSUE LOG

Column : has_momo_account
  Issue      : 7+ inconsistent encodings (yes, Y, 1, TRUE, No, N, 0)
  Resolution : standardize_boolean() → 0/1/NaN
  Rationale  : Preserve NaN (missing ≠ No). Strip whitespace first.

Column : annual_revenue_ghs
  Issue      : Mixed currency prefixes: "GHS 37,586.71", "$1136.69"
  Resolution : clean_currency() → regex strip → float
  Rationale  : Do not assume USD→GHS conversion rate; strip numerically

Column : owner_gender
  Issue      : Inconsistent (Male, male, M, m, Female, female, F, f)
  Resolution : standardize_gender() → canonical Male/Female/NaN
  Rationale  : Kept for fairness audit; NOT a model feature

Column : gra_tin
  Issue      : Many empty, PENDING, N/A values; rest are valid TINs
  Resolution : create_has_tin() → binary 0/1; drop raw column
  Rationale  : TIN number has no predictive meaning; HAVING one does

Column : owner_age
  Issue      : Values as low as 14 — invalid (Ghana legal minimum = 18)
  Resolution : val

In [4]:
# ── CELL 4: Before-cleaning audit ─────────────────────────────────────────────
# Snapshot of raw data issues for comparison after cleaning
print('BEFORE CLEANING — Snapshot of key columns:')
print()
print('has_momo_account unique values:')
print(df_raw['has_momo_account'].value_counts(dropna=False).to_string())
print()
print('annual_revenue_ghs sample (showing format issues):')
print(df_raw['annual_revenue_ghs'].head(20).to_list())
print()
print('owner_gender unique values:')
print(df_raw['owner_gender'].value_counts(dropna=False).to_string())
print()
print('previous_default value counts:')
print(df_raw['previous_default'].value_counts(dropna=False).to_string())
print()
print(f'credit_bureau_score missing: {df_raw["credit_bureau_score"].isnull().mean():.1%}')
print(f'owner_age invalid (<18): {(df_raw["owner_age"] < 18).sum()} rows')

BEFORE CLEANING — Snapshot of key columns:

has_momo_account unique values:
has_momo_account
Yes     845
No      457
yes     338
Y       321
1       301
0       291
TRUE    177
N       159
no      148

annual_revenue_ghs sample (showing format issues):
['11218.55', '5620.4', '39586.37', '45732.04', '48242.71', '6336.94', '9541.63', '193829.36', '15858', nan, '23832', '129909.72', '140057.94', '59894.26', '11105.63', '98530.58', '88534.73', '272547.71', '8955.76', '11331.87']

owner_gender unique values:
owner_gender
Male      899
Female    726
female    514
M         301
F         300
male      297

previous_default value counts:
previous_default
No     2079
NaN     604
Yes     354

credit_bureau_score missing: 64.8%
owner_age invalid (<18): 30 rows


## Step 2: Apply All Cleaning Functions

In [5]:
# ── CELL 5: Run clean_dataframe() ─────────────────────────────────────────────
# All 9 data quality fixes in one call.
# Every fix is implemented in src/preprocessing.py — same code runs at inference.

df_cleaned = clean_dataframe(df_raw)

print(f'\nShape after cleaning: {df_cleaned.shape}')
print(f'Rows preserved: {len(df_cleaned):,} of {len(df_raw):,} ({len(df_cleaned)/len(df_raw):.1%})')

clean_dataframe: 3,037 rows processed, 5253 total null values remaining

Shape after cleaning: (3037, 39)
Rows preserved: 3,037 of 3,037 (100.0%)


## Step 3: Verify Each Fix

In [6]:
# ── CELL 6: Verify Fix 1 — has_momo_account ───────────────────────────────────
print('FIX 1: has_momo_account')
print('Before:', df_raw['has_momo_account'].value_counts(dropna=False).to_dict())
print('After: ', df_cleaned['has_momo_account'].value_counts(dropna=False).to_dict())
assert df_cleaned['has_momo_account'].isin([0, 1, np.nan]).all() or \
       df_cleaned['has_momo_account'].isin([0.0, 1.0]).all() or \
       df_cleaned['has_momo_account'].dropna().isin([0, 1]).all(), \
       'has_momo_account still has unexpected values!'
print('PASS: only 0, 1, NaN')

FIX 1: has_momo_account
Before: {'Yes': 845, 'No': 457, 'yes': 338, 'Y': 321, '1': 301, '0': 291, 'TRUE': 177, 'N': 159, 'no': 148}
After:  {1: 1982, 0: 1055}
PASS: only 0, 1, NaN


In [7]:
# ── CELL 7: Verify Fix 2 — annual_revenue_ghs ─────────────────────────────────
print('FIX 2: annual_revenue_ghs')
print(f'dtype: {df_cleaned["annual_revenue_ghs"].dtype}')
print(f'Non-null count: {df_cleaned["annual_revenue_ghs"].notnull().sum()}')
print(f'Min: {df_cleaned["annual_revenue_ghs"].min():,.2f}')
print(f'Max: {df_cleaned["annual_revenue_ghs"].max():,.2f}')
print(f'Missing: {df_cleaned["annual_revenue_ghs"].isnull().sum()}')
assert df_cleaned['annual_revenue_ghs'].dtype in ['float64', 'float32'], \
    'annual_revenue_ghs is not numeric!'
print('PASS: numeric dtype')

FIX 2: annual_revenue_ghs
dtype: float64
Non-null count: 2950
Min: 102.61
Max: 9,498,575.16
Missing: 87
PASS: numeric dtype


In [8]:
# ── CELL 8: Verify Fix 3 — owner_gender ───────────────────────────────────────
print('FIX 3: owner_gender → owner_gender_clean')
print(df_cleaned['owner_gender_clean'].value_counts(dropna=False).to_string())
valid_genders = {'Male', 'Female', float('nan')}
unique_genders = set(df_cleaned['owner_gender_clean'].dropna().unique())
assert unique_genders.issubset({'Male', 'Female'}), \
    f'Unexpected gender values: {unique_genders}'
print('PASS: only Male, Female, NaN')

FIX 3: owner_gender → owner_gender_clean
owner_gender_clean
Female    1540
Male      1497
PASS: only Male, Female, NaN


In [9]:
# ── CELL 9: Verify Fix 4 — has_tin ────────────────────────────────────────────
print('FIX 4: gra_tin → has_tin')
print(f'has_tin distribution: {df_cleaned["has_tin"].value_counts().to_dict()}')
print(f'  No TIN (0): {(df_cleaned["has_tin"]==0).sum():,} applicants ({(df_cleaned["has_tin"]==0).mean():.1%})')
print(f'  Has TIN (1): {(df_cleaned["has_tin"]==1).sum():,} applicants ({(df_cleaned["has_tin"]==1).mean():.1%})')
print()
tin_default = df_cleaned.groupby('has_tin')[TARGET].mean()
print('Default rate by TIN status (validates predictive signal):')
print(tin_default.to_string())
print('PASS: binary 0/1')

FIX 4: gra_tin → has_tin
has_tin distribution: {1: 1898, 0: 1139}
  No TIN (0): 1,139 applicants (37.5%)
  Has TIN (1): 1,898 applicants (62.5%)

Default rate by TIN status (validates predictive signal):
has_tin
0    0.122915
1    0.138567
PASS: binary 0/1


In [10]:
# ── CELL 10: Verify Fix 5 — owner_age ─────────────────────────────────────────
print('FIX 5: owner_age — invalid ages (<18) set to NaN')
invalid_before = (df_raw['owner_age'] < 18).sum()
invalid_after  = (df_cleaned['owner_age'] < 18).sum()
print(f'  Invalid ages before: {invalid_before}')
print(f'  Invalid ages after:  {invalid_after} (should be 0)')
print(f'  Min age now: {df_cleaned["owner_age"].min()}')
print(f'  Missing ages: {df_cleaned["owner_age"].isnull().sum()}')
assert invalid_after == 0, 'Still have ages < 18!'
print('PASS')

FIX 5: owner_age — invalid ages (<18) set to NaN
  Invalid ages before: 30
  Invalid ages after:  0 (should be 0)
  Min age now: 18.0
  Missing ages: 41
PASS


In [11]:
# ── CELL 11: Verify Fix 6 — previous_default ──────────────────────────────────
print('FIX 6: previous_default → two features')
print('\nhas_previous_loan_history:')
print(df_cleaned['has_previous_loan_history'].value_counts().to_string())
print('\nprevious_default_clean:')
print(df_cleaned['previous_default_clean'].value_counts().to_string())
assert set(df_cleaned['previous_default_clean'].unique()).issubset({'Yes', 'No', 'Unknown'}), \
    'Unexpected values in previous_default_clean'
print('PASS: Yes/No/Unknown, no nulls')

FIX 6: previous_default → two features

has_previous_loan_history:
has_previous_loan_history
1    2433
0     604

previous_default_clean:
previous_default_clean
No         2079
Unknown     604
Yes         354
PASS: Yes/No/Unknown, no nulls


In [12]:
# ── CELL 12: Verify Fix 7 — credit_bureau_score ───────────────────────────────
print('FIX 7: credit_bureau_score → flag + imputed')
print(f'has_credit_bureau_score: {df_cleaned["has_credit_bureau_score"].value_counts().to_dict()}')
print(f'credit_bureau_score_imputed — missing: {df_cleaned["credit_bureau_score_imputed"].isnull().sum()}')
print(f'  Range: {df_cleaned["credit_bureau_score_imputed"].min():.0f} – {df_cleaned["credit_bureau_score_imputed"].max():.0f}')
print()
print('Default rate by bureau score presence:')
print(df_cleaned.groupby('has_credit_bureau_score')[TARGET].mean().to_string())
print('PASS')

FIX 7: credit_bureau_score → flag + imputed
has_credit_bureau_score: {0: 1967, 1: 1070}
credit_bureau_score_imputed — missing: 0
  Range: 103 – 876

Default rate by bureau score presence:
has_credit_bureau_score
0    0.131164
1    0.135514
PASS


In [13]:
# ── CELL 13: Verify Fix 8 — tenure_is_zero_flag ───────────────────────────────
print('FIX 8: bank_account_tenure_months → tenure_is_zero_flag')
print(f'Zero-tenure accounts flagged: {df_cleaned["tenure_is_zero_flag"].sum()}')
print(f'tenure_is_zero_flag distribution: {df_cleaned["tenure_is_zero_flag"].value_counts().to_dict()}')
print('NOTE: Original tenure column preserved — zeros not nulled.')
print('PASS')

FIX 8: bank_account_tenure_months → tenure_is_zero_flag
Zero-tenure accounts flagged: 76
tenure_is_zero_flag distribution: {0: 2961, 1: 76}
NOTE: Original tenure column preserved — zeros not nulled.
PASS


In [14]:
# ── CELL 14: Verify Fix 9 — date features ─────────────────────────────────────
print('FIX 9: application_date → cyclical month features + year')
print(f'app_month_sin range: [{df_cleaned["app_month_sin"].min():.3f}, {df_cleaned["app_month_sin"].max():.3f}]')
print(f'app_month_cos range: [{df_cleaned["app_month_cos"].min():.3f}, {df_cleaned["app_month_cos"].max():.3f}]')
print(f'app_year unique values: {sorted(df_cleaned["app_year"].unique())}')
print('WHY sin/cos: month 12 and month 1 are adjacent — sin/cos encodes this circularity.')
print('PASS')

FIX 9: application_date → cyclical month features + year
app_month_sin range: [-1.000, 1.000]
app_month_cos range: [-1.000, 1.000]
app_year unique values: [np.int64(2024), np.int64(2025)]
WHY sin/cos: month 12 and month 1 are adjacent — sin/cos encodes this circularity.
PASS


## Step 4: Post-Cleaning Summary

In [15]:
# ── CELL 15: Missing data before vs after ─────────────────────────────────────
missing_before = df_raw.isnull().sum().sum()
missing_after  = df_cleaned.isnull().sum().sum()

print(f'Total null values BEFORE cleaning: {missing_before:,}')
print(f'Total null values AFTER cleaning:  {missing_after:,}')
print(f'Remaining nulls are handled by the sklearn Pipeline imputer.')
print()

# ALL_MODEL_FEATURES includes engineered columns (log transforms, ratios) that are
# created in notebook 03. Only check columns that exist at this stage.
base_features_present = [f for f in ALL_MODEL_FEATURES if f in df_cleaned.columns]
engineered_pending    = [f for f in ALL_MODEL_FEATURES if f not in df_cleaned.columns]

remaining = df_cleaned[base_features_present].isnull().sum()
remaining = remaining[remaining > 0].sort_values(ascending=False)
print('REMAINING MISSING IN BASE FEATURES (to be imputed by sklearn Pipeline):')
print(remaining.to_string())
print()
print(f'Engineered features (created in notebook 03 — not checked here): {len(engineered_pending)}')
print(engineered_pending)

Total null values BEFORE cleaning: 5,212
Total null values AFTER cleaning:  5,253
Remaining nulls are handled by the sklearn Pipeline imputer.

REMAINING MISSING IN BASE FEATURES (to be imputed by sklearn Pipeline):
collateral_type    367
owner_age           41

Engineered features (created in notebook 03 — not checked here): 22
['log_annual_revenue_ghs', 'log_monthly_momo_volume_ghs', 'log_avg_monthly_bank_balance_ghs', 'log_loan_amount_requested_ghs', 'log_collateral_value_ghs', 'loan_to_revenue_ratio', 'collateral_coverage_ratio', 'revenue_per_employee', 'momo_to_revenue_ratio', 'bureau_x_coverage', 'revenue_per_year_log', 'momo_annual_to_loan', 'bank_balance_to_loan_ratio', 'previous_default_numeric', 'formality_score', 'sector_x_region', 'high_leverage_flag', 'good_collateral_flag', 'has_momo_and_tin', 'repeat_borrower_flag', 'credit_score_risk_flag', 'unsecured_loan_flag']


In [16]:
# ── CELL 16: Confirm data leakage columns still present (for logging) ──────────
# We will drop rm_recommendation and internal_risk_grade in the NEXT step.
# Here we do a final look at what they contain before dropping.
print('DATA LEAKAGE COLUMNS — final inspection before dropping:')
print()
print('rm_recommendation:')
print(df_cleaned['rm_recommendation'].value_counts())
print()
print('internal_risk_grade:')
print(df_cleaned['internal_risk_grade'].value_counts())
print()
print('WHY dropping these is non-negotiable:')
print('  At inference time (new application), neither column exists.')
print('  Training with them teaches the model to mimic a human reviewer,')
print('  not to discover patterns in the underlying financial data.')
print('  In production, these features are NULL → model would crash.')

DATA LEAKAGE COLUMNS — final inspection before dropping:

rm_recommendation:
rm_recommendation
Approve    2187
Decline     426
Refer       424
Name: count, dtype: int64

internal_risk_grade:
internal_risk_grade
B    1184
A     942
C     559
D     214
E     138
Name: count, dtype: int64

WHY dropping these is non-negotiable:
  At inference time (new application), neither column exists.
  Training with them teaches the model to mimic a human reviewer,
  not to discover patterns in the underlying financial data.
  In production, these features are NULL → model would crash.


In [17]:
# ── CELL 17: Save cleaned data to interim/ ────────────────────────────────────
# We save the FULL cleaned dataframe (before dropping any columns).
# This preserves rm_recommendation and internal_risk_grade for audit logging
# and fairness columns for notebook 07.
# Feature selection and drops happen in notebook 03 and the sklearn Pipeline.

INTERIM_PATH = f'{BASE_PATH}/data/interim/cleaned_pre_split.csv'
df_cleaned.to_csv(INTERIM_PATH, index=False)

print(f'Saved: {INTERIM_PATH}')
print(f'Shape: {df_cleaned.shape}')
print(f'Columns: {list(df_cleaned.columns)}')
print()
print('Proceed to notebook 03_feature_engineering.ipynb')

Saved: /content/drive/MyDrive/stanbic_sme_credit/data/interim/cleaned_pre_split.csv
Shape: (3037, 39)
Columns: ['application_id', 'business_name', 'sector', 'region', 'ethnic_group', 'years_in_operation', 'owner_gender', 'owner_age', 'disability_status', 'num_employees', 'annual_revenue_ghs', 'monthly_momo_volume_ghs', 'avg_monthly_bank_balance_ghs', 'bank_account_tenure_months', 'has_momo_account', 'gra_tin', 'loan_amount_requested_ghs', 'loan_purpose', 'collateral_type', 'collateral_value_ghs', 'previous_loan_count', 'previous_default', 'credit_bureau_score', 'rm_recommendation', 'internal_risk_grade', 'application_date', 'contact_phone', 'days_past_due_current', 'loan_default', 'owner_gender_clean', 'has_tin', 'has_previous_loan_history', 'previous_default_clean', 'has_credit_bureau_score', 'credit_bureau_score_imputed', 'tenure_is_zero_flag', 'app_month_sin', 'app_month_cos', 'app_year']

Proceed to notebook 03_feature_engineering.ipynb
